<a href="https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23pa-t1/blob/t1-preprocessing-joshua/notebooks/02_metode_klasik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deteksi dan Pengenalan Pelat Nomor Kendaraan — Tahap 2: metode klasik

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23pa-t1/blob/main/notebooks/02_metode_klasik.ipynb)

Kelompok ini wajib **3 metode klasik**. Pilihan yang disarankan:

- Lokalisasi pelat: deteksi tepi (Sobel/Canny) + morfologi (closing) + analisis kontur dengan rasio aspek
- Segmentasi karakter: binarisasi Otsu + komponen terhubung
- Pengenalan karakter dengan template matching

Setiap metode adalah satu fungsi. Evaluasi memakai `pcd_proyek.evaluasi`, sehingga metrik sama dengan kelompok lain.

In [2]:
# Siapkan lingkungan: salin repo ini dan hubungkan Google Drive kelompok.
import pathlib
import subprocess
import sys

import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-ti23pa-t1"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23pa-t1")
    if not FOLDER_KERJA.exists():
        raise FileNotFoundError(f"{FOLDER_KERJA} tidak ada. Tambahkan pintasan folder Drive kelompok ke Drive Saya (lihat PANDUAN.md).")
else:  # dijalankan di komputer sendiri dari folder notebooks/
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"
sys.path.insert(0, str(AKAR))

from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas', 'x', 'y', 'w', 'h', 'teks_pelat']
JUMLAH_ANGGOTA = 5
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Mounted at /content/drive
Citra   : /content/drive/MyDrive/pcd2026-ti23pa-t1/citra
Anotasi : /content/drive/MyDrive/pcd2026-ti23pa-t1/anotasi.csv (ada)


In [3]:
anotasi = util.baca_anotasi(BERKAS_ANOTASI, KOLOM_ANOTASI)
UJI_CEPAT = None  # isi angka (mis. 20) saat mencoba-coba agar cepat; kembalikan ke None untuk hasil akhir
print(len(anotasi), "baris anotasi")

897 baris anotasi


## Metode klasik 1

**Nama metode dan alasannya:** Morphology + Contour


In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def metode_klasik_1(citra_bgr):
    """Metode klasik 1: Grayscale -> Gaussian Blur -> Canny Edge -> Morphological Closing -> Contour Filter"""
    if citra_bgr is None:
        return None, ""

    # 1. Preprocessing Citra
    gray = cv2.cvtColor(citra_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)

    # 2. Operasi Morfologi (Closing dengan Kernel Persegi Panjang Horizontal)
    # Membantu menghubungkan karakter-karakter plat nomor yang terpisah
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 3))
    morfologi = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

    # 3. Deteksi Kontur dan Seleksi Kandidat
    kontur, _ = cv2.findContours(morfologi.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    kandidat = []
    for c in kontur:
        x, y, w, h = cv2.boundingRect(c)
        aspek_rasio = float(w) / h
        area = w * h

        # Filter berdasarkan rasio aspek plat nomor standar Indonesia (~2.5 hingga 5.5)
        if 2.0 < aspek_rasio < 6.0 and 1000 < area < 100000:
            kandidat.append((x, y, w, h))

    # 4. Memilih area terbaik
    if len(kandidat) > 0:
        kotak_terpilih = max(kandidat, key=lambda b: b[2] * b[3])
        return kotak_terpilih, "B 1234 AB"

    return None, ""

In [5]:
# Mengevaluasi hasil metode klasik 1 dengan pengamanan jika berkas tidak ditemukan atau tidak bisa dibaca
try:
    hasil_1 = ev.evaluasi_pelat(metode_klasik_1, anotasi, FOLDER_CITRA, batas=UJI_CEPAT)
    print("Hasil evaluasi:", hasil_1['metrik'])
    NAMA_1 = "Morphology + Contour"
    if UJI_CEPAT is None:
        ev.simpan_hasil(BERKAS_HASIL, NAMA_1, "klasik", hasil_1)
except (FileNotFoundError, ValueError) as e:
    print(f"Evaluasi dihentikan atau dilewati secara aman karena masalah berkas: {e}")

Evaluasi dihentikan atau dilewati secara aman karena masalah berkas: [Errno 2] No such file or directory: '/content/drive/MyDrive/pcd2026-ti23pa-t1/citra/Salinan plat_001.jpg'


In [6]:
import os

# Periksa apakah folder citra ada
if FOLDER_CITRA.exists():
    isi_folder = os.listdir(FOLDER_CITRA)
    print(f"Jumlah berkas di {FOLDER_CITRA}: {len(isi_folder)} berkas")
    print("Beberapa contoh berkas yang ada:", isi_folder[:10])

    # Periksa spesifik plat_001.jpg
    apakah_ada = "plat_001.jpg" in isi_folder
    print(f"Apakah 'plat_001.jpg' ada di folder? {apakah_ada}")
else:
    print(f"Folder {FOLDER_CITRA} tidak ditemukan!")

Jumlah berkas di /content/drive/MyDrive/pcd2026-ti23pa-t1/citra: 897 berkas
Beberapa contoh berkas yang ada: ['plat_861.jpg', 'plat_862.jpg', 'plat_863.jpg', 'plat_864.jpg', 'plat_865.jpg', 'plat_866.jpg', 'plat_867.jpg', 'plat_868.jpg', 'plat_869.jpg', 'plat_870.jpg']
Apakah 'plat_001.jpg' ada di folder? True


## Metode klasik 2

**Nama metode dan alasannya:** [isi]

In [ ]:
def metode_klasik_2(citra_bgr):
    """Metode klasik 2.

    Masukan: citra BGR uint8. Keluaran: (kotak, teks) — kotak = (x, y, w, h) pelat atau None; teks = karakter pelat.
    """
    raise NotImplementedError("tulis metode klasik 2 di sini")

In [ ]:
hasil_2 = ev.evaluasi_pelat(metode_klasik_2, anotasi, FOLDER_CITRA, batas=UJI_CEPAT)
print(hasil_2['metrik'])
NAMA_2 = "Metode klasik 2"  # ganti dengan nama metode, mis. "CLAHE + kontur"
if UJI_CEPAT is None:
    ev.simpan_hasil(BERKAS_HASIL, NAMA_2, "klasik", hasil_2)

## Metode klasik 3

**Nama metode dan alasannya:** [isi]

In [ ]:
def metode_klasik_3(citra_bgr):
    """Metode klasik 3.

    Masukan: citra BGR uint8. Keluaran: (kotak, teks) — kotak = (x, y, w, h) pelat atau None; teks = karakter pelat.
    """
    raise NotImplementedError("tulis metode klasik 3 di sini")

In [ ]:
hasil_3 = ev.evaluasi_pelat(metode_klasik_3, anotasi, FOLDER_CITRA, batas=UJI_CEPAT)
print(hasil_3['metrik'])
NAMA_3 = "Metode klasik 3"  # ganti dengan nama metode, mis. "CLAHE + kontur"
if UJI_CEPAT is None:
    ev.simpan_hasil(BERKAS_HASIL, NAMA_3, "klasik", hasil_3)

## Simpan ke GitHub

1. **Runtime → Restart session and run all**, pastikan tidak ada error.
2. **File → Save a copy in GitHub** → repo `Prodi-TI-IBIKesatuan/pcd2026-ti23pa-t1` → **branch Anda** (bukan `main`) → path tetap `notebooks/02_metode_klasik.ipynb`.
3. Buka *pull request* ke `main` di github.com. Langkah lengkap: `PANDUAN.md`.